In [ ]:
noise_file = "/sw/vis2/shaheen3/visit-src-3.4.2/install/data/noise.silo"

In [ ]:
import os
os.environ["VTK_DEFAULT_OPENGL_WINDOW"] = "vtkOSOpenGLRenderWindow"

from paraview.simple import *
import paraview.servermanager as sm
from IPython.display import Image, display

if not (sm.ActiveConnection and sm.ActiveConnection.IsRemote()):
    print("🔄 Connecting to remote pvserver...")
    Connect("127.0.0.1", 11111)
    print("✅ Connected to pvserver")
else:
    print("✅ Already connected to pvserver")


In [ ]:
def reset_pipeline():
    # Delete all sources
    for _, src in list(GetSources().items()):
        Delete(src)

    # Delete all views
    for v in list(GetViews()):
        Delete(v)

    print("♻️ Pipeline reset (server preserved)")


In [ ]:
from paraview.simple import *
from IPython.display import Image, display

# Clean pipeline only (do NOT disconnect)
for _, src in list(GetSources().items()):
    Delete(src)
for v in list(GetViews()):
    Delete(v)

# --- Load noise.silo ---
noise = VisItSiloReader(registrationName='noise.silo',
    FileName=[noise_file]
)

# THIS IS THE CRITICAL PART
noise.Set(
    MeshStatus=['Mesh', 'Mesh2D', 'PointMesh'],
    MaterialStatus=['1 air', '2 chrome'],
    CellArrayStatus=['airVf', 'airVfGradient', 'chromeVf'],
    PointArrayStatus=['PointVar', 'grad', 'hardyglobal', 'hgslice', 'radial', 'shepardglobal', 'tensor_comps/grad_tensor_ii', 'tensor_comps/grad_tensor_ij', 'tensor_comps/grad_tensor_ik', 'tensor_comps/grad_tensor_ji', 'tensor_comps/grad_tensor_jj', 'tensor_comps/grad_tensor_jk', 'tensor_comps/grad_tensor_ki', 'tensor_comps/grad_tensor_kj', 'tensor_comps/grad_tensor_kk', 'x'],
)


noise.UpdatePipeline()

# --- Visualization ---
view = CreateView("RenderView")
rep = Show(noise, view)

ColorBy(rep, ("POINTS", "hardyglobal"))
rep.SetRepresentationType("Surface")

lut = GetColorTransferFunction("hardyglobal")
lut.ApplyPreset("Blue to Red Rainbow", True)
lut.RescaleTransferFunctionToDataRange(True)

ResetCamera()
Render()

# --- Save & display ---
fname = "/tmp/noise_silo.png"
SaveScreenshot(fname, view, ImageResolution=[800, 600])
display(Image(filename=fname))

print("✅ noise.silo rendered correctly")
